# 05 — Sistema de Recomendação Colaborativa

## 1. Imports e Configurações

In [1]:
import sqlite3
import os
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

DB_PATH     = '/home/vinic/Documentos/Projetos/LH_Nautical/datasets/nautical.db'
MODELS_PATH = '/home/vinic/Documentos/Projetos/LH_Nautical/models'

os.makedirs(MODELS_PATH, exist_ok=True)
print(f'DB     : {DB_PATH}')
print(f'Models : {MODELS_PATH}')

DB     : /home/vinic/Documentos/Projetos/LH_Nautical/datasets/nautical.db
Models : /home/vinic/Documentos/Projetos/LH_Nautical/models


## 2. Carregando os Dados

In [2]:
conn = sqlite3.connect(DB_PATH)

df_vendas   = pd.read_sql('SELECT id_client, id_product, qtd, total FROM vendas', conn)
df_produtos = pd.read_sql('SELECT code, name, category FROM produtos', conn)
df_clientes = pd.read_sql('SELECT code, full_name, city, state FROM clientes', conn)

conn.close()

print(f'Vendas   : {df_vendas.shape}')
print(f'Produtos : {df_produtos.shape}')
print(f'Clientes : {df_clientes.shape}')
print(f'\nClientes únicos nas vendas : {df_vendas["id_client"].nunique()}')
print(f'Produtos únicos nas vendas : {df_vendas["id_product"].nunique()}')
df_vendas.head()

Vendas   : (9895, 4)
Produtos : (157, 3)
Clientes : (49, 4)

Clientes únicos nas vendas : 49
Produtos únicos nas vendas : 150


,id_client,id_product,qtd,total
0,42,105,11,3405.0
1,3,136,9,16873.9
2,25,139,7,9475.3
3,20,23,5,55893.0
4,8,57,4,451403.9


## 3. Construindo a Matriz Cliente × Produto

Cada célula indica **se o cliente comprou** (1) **ou não** (0) o produto.
Focado na presença binária, não quantidade — o foco é *o que* foi comprado, não *quanto*.

In [3]:
# Agregar: 1 se o cliente comprou o produto (independente de quantas vezes)
compras = (
    df_vendas.groupby(['id_client', 'id_product'])
    .size()
    .reset_index(name='comprado')
)
compras['comprado'] = 1  # binário

# Pivot: linhas = clientes, colunas = produtos
matriz = compras.pivot_table(
    index='id_client',
    columns='id_product',
    values='comprado',
    fill_value=0
)

print(f'Matriz shape : {matriz.shape}  (clientes × produtos)')
print(f'Esparsidade  : {1 - matriz.values.sum() / matriz.size:.1%} de zeros')
print(f'\nMédia de produtos por cliente : {matriz.sum(axis=1).mean():.1f}')
print(f'Máx  de produtos por cliente  : {matriz.sum(axis=1).max()}')
print(f'Mín  de produtos por cliente  : {matriz.sum(axis=1).min()}')
matriz.iloc[:5, :8]

Matriz shape : (49, 150)  (clientes × produtos)
Esparsidade  : 26.4% de zeros

Média de produtos por cliente : 110.5
Máx  de produtos por cliente  : 124.0
Mín  de produtos por cliente  : 95.0


id_product,1,2,3,4,5,6,7,8
id_client,,,,,,,,
1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
5,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [4]:
# Converter para matriz esparsa (eficiente em memória e cálculo)
matriz_sparse = csr_matrix(matriz.values)
print(f'Matriz esparsa: {matriz_sparse.shape}, {matriz_sparse.nnz} elementos não-zero')

Matriz esparsa: (49, 150), 5413 elementos não-zero


## 4. Treinando o Modelo KNN

In [5]:
N_NEIGHBORS = 10  # número de vizinhos para buscar

modelo = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

modelo.fit(matriz_sparse)
print(f'Modelo treinado com {modelo.n_samples_fit_} clientes.')

Modelo treinado com 49 clientes.


## 5. Função de Recomendação

In [6]:
def recomendar_produtos(
    id_client: int,
    modelo: NearestNeighbors,
    matriz: pd.DataFrame,
    df_produtos: pd.DataFrame,
    df_clientes: pd.DataFrame,
    n_recomendacoes: int = 5,
    n_vizinhos: int = 10
) -> pd.DataFrame:
    """
    Retorna um DataFrame com os produtos recomendados para um cliente.

    Parâmetros
    ----------
    id_client        : código do cliente
    modelo           : NearestNeighbors já treinado
    matriz           : DataFrame cliente × produto (binário)
    df_produtos      : tabela de produtos
    df_clientes      : tabela de clientes
    n_recomendacoes  : quantos produtos recomendar
    n_vizinhos       : quantos vizinhos usar para agregar votos

    Retorna
    -------
    DataFrame com colunas: id_produto, nome, categoria, score
    """
    if id_client not in matriz.index:
        raise ValueError(f'Cliente {id_client} não encontrado na matriz.')

    # Vetor de compras do cliente alvo
    idx_cliente = matriz.index.get_loc(id_client)
    vetor_cliente = matriz_sparse[idx_cliente]

    # Buscar vizinhos (inclui o próprio cliente, por isso +1)
    distancias, indices = modelo.kneighbors(
        vetor_cliente,
        n_neighbors=min(n_vizinhos + 1, modelo.n_samples_fit_)
    )

    # Remover o próprio cliente dos vizinhos
    ids_vizinhos = [
        matriz.index[i] for i in indices[0]
        if matriz.index[i] != id_client
    ][:n_vizinhos]
    sims_vizinhos = [
        1 - d for i, d in zip(indices[0], distancias[0])
        if matriz.index[i] != id_client
    ][:n_vizinhos]

    # Produtos que o cliente alvo JÁ comprou
    ja_comprados = set(matriz.columns[matriz.loc[id_client] > 0])

    # Agregar score: soma ponderada pela similaridade de cada vizinho
    scores = {}
    for vizinho, sim in zip(ids_vizinhos, sims_vizinhos):
        if sim <= 0:
            continue
        produtos_vizinho = set(matriz.columns[matriz.loc[vizinho] > 0])
        novos = produtos_vizinho - ja_comprados
        for prod in novos:
            scores[prod] = scores.get(prod, 0) + sim

    if not scores:
        print(f'Nenhuma recomendação encontrada para o cliente {id_client}.')
        return pd.DataFrame()

    # Montar resultado
    df_scores = pd.Series(scores, name='score').sort_values(ascending=False)
    top = df_scores.head(n_recomendacoes).reset_index()
    top.columns = ['id_produto', 'score']

    # Enriquecer com nome e categoria
    top = top.merge(
        df_produtos.rename(columns={'code': 'id_produto', 'name': 'nome', 'category': 'categoria'}),
        on='id_produto', how='left'
    )

    return top[['id_produto', 'nome', 'categoria', 'score']]


def info_cliente(id_client, df_clientes, df_vendas, df_produtos):
    """Exibe resumo do cliente e seus produtos já comprados."""
    cli = df_clientes[df_clientes['code'] == id_client].iloc[0]
    print(f'\n{'='*60}')
    print(f'  Cliente #{id_client}: {cli["full_name"]}')
    print(f'  Local: {cli["city"]} / {cli["state"]}')
    compras_cli = df_vendas[df_vendas['id_client'] == id_client]['id_product'].unique()
    prods = df_produtos[df_produtos['code'].isin(compras_cli)]
    print(f'  Produtos já comprados ({len(compras_cli)}):')
    for _, r in prods.iterrows():
        print(f'    [{r["category"]}] {r["name"]}')
    print('='*60)

## 6. Testando para 3 Clientes Diferentes

In [7]:
# Escolher 3 clientes com perfis distintos:
# - um com muitas compras, um mediano, um com poucas
compras_por_cliente = df_vendas.groupby('id_client')['id_product'].nunique().sort_values(ascending=False)

# Garantir que estão na matriz (clientes com ao menos 1 compra)
clientes_validos = [c for c in compras_por_cliente.index if c in matriz.index]

n_total = len(clientes_validos)
cliente_heavy  = clientes_validos[0]                     # mais compras
cliente_medio  = clientes_validos[n_total // 2]          # mediano
cliente_light  = clientes_validos[-1]                    # menos compras

clientes_teste = [cliente_heavy, cliente_medio, cliente_light]
print('Clientes selecionados para teste:')
for c in clientes_teste:
    n = compras_por_cliente[c]
    nome = df_clientes[df_clientes['code'] == c]['full_name'].values[0]
    print(f'  ID {c:>3} | {n:>2} produtos | {nome}')

Clientes selecionados para teste:
  ID  15 | 124 produtos | Carla Lopes Alves Pacheco Rocha
  ID  28 | 110 produtos | Bianca Rodrigues
  ID  31 | 95 produtos | Jéssica Figueiredo Leite Martins


In [8]:
# ---- Teste Cliente 1 (comprador pesado) ----
info_cliente(cliente_heavy, df_clientes, df_vendas, df_produtos)
rec1 = recomendar_produtos(cliente_heavy, modelo, matriz, df_produtos, df_clientes, n_recomendacoes=5)
print('\n>> Recomendações:')
print(rec1.to_string(index=False))


  Cliente #15: Carla Lopes Alves Pacheco Rocha
  Local: Fortaleza do Tabocão / TO
  Produtos já comprados (124):
    [ELETRONICOS] Transponder AIS Maré Magnum
    [ELETRONICOS] Transponder Furuno Marlin
    [ELETRONICOS] Radar Furuno Pulse Leviathan
    [ELETRONICOS] Rádio AIS Hydro Tidal Zen
    [ELETRONICOS] Piloto Automático Furuno Storm
    [ELETRONICOS] Transponder AIS Vector
    [ELETRONICOS] GPS AIS Zen
    [ELETRONICOS] Transponder AIS Titan Pulse
    [ELETRONICOS] Piloto Automático Simrad Titan Flux Magnum
    [ELETRONICOS] GPS Furuno Swift Leviathan Poseidon
    [ELETRONICOS] GPS AIS Drift Flux Vox
    [ELETRONICOS] Rádio Furuno Zen Swift
    [ELETRONICOS] Radar Garmin Tidal Thrust
    [ELETRONICOS] Sonda Garmin Axis
    [ELETRONICOS] Sonda Lowrance Tidal Storm Vox
    [ELETRONICOS] GPS AIS Nautic Thrust
    [ELETRONICOS] Transponder Furuno Force Ion
    [ELETRONICOS] Piloto Automático Furuno Torque Peak
    [ELETRONICOS] GPS AIS Orca Marlin Velocity
    [ELETRONICOS] Rádio 

In [9]:
# ---- Teste Cliente 2 (perfil mediano) ----
info_cliente(cliente_medio, df_clientes, df_vendas, df_produtos)
rec2 = recomendar_produtos(cliente_medio, modelo, matriz, df_produtos, df_clientes, n_recomendacoes=5)
print('\n>> Recomendações:')
print(rec2.to_string(index=False))


  Cliente #28: Bianca Rodrigues
  Local: Antonina / PR
  Produtos já comprados (110):
    [ELETRONICOS] Radar Furuno Pulse Leviathan
    [ELETRONICOS] Rádio AIS Hydro Tidal Zen
    [ELETRONICOS] Transponder AIS Vector
    [ELETRONICOS] GPS AIS Zen
    [ELETRONICOS] Transponder AIS Titan Pulse
    [ELETRONICOS] Piloto Automático Simrad Titan Flux Magnum
    [ELETRONICOS] GPS AIS Drift Flux Vox
    [ELETRONICOS] Radar Simrad Boost
    [ELETRONICOS] Rádio Furuno Zen Swift
    [ELETRONICOS] Radar Garmin Tidal Thrust
    [ELETRONICOS] Rádio Simrad Axis
    [ELETRONICOS] Sonda Lowrance Tidal Storm Vox
    [ELETRONICOS] GPS AIS Nautic Thrust
    [ELETRONICOS] Piloto Automático Furuno Torque Peak
    [ELETRONICOS] GPS AIS Orca Marlin Velocity
    [ELETRONICOS] Rádio Lowrance Nitro Thrust Barracuda
    [ELETRONICOS] Piloto Automático Furuno Core Boost Flux
    [ELETRONICOS] GPS Garmin Vortex Maré Drift
    [ELETRONICOS] Piloto Automático AIS Core
    [ELETRONICOS] Transponder AIS Boost
    [EL

In [10]:
# ---- Teste Cliente 3 (comprador leve) ----
info_cliente(cliente_light, df_clientes, df_vendas, df_produtos)
rec3 = recomendar_produtos(cliente_light, modelo, matriz, df_produtos, df_clientes, n_recomendacoes=5)
print('\n>> Recomendações:')
print(rec3.to_string(index=False))


  Cliente #31: Jéssica Figueiredo Leite Martins
  Local: São Luís / MA
  Produtos já comprados (95):
    [ELETRONICOS] Transponder AIS Maré Magnum
    [ELETRONICOS] Radar Furuno Pulse Leviathan
    [ELETRONICOS] Rádio AIS Hydro Tidal Zen
    [ELETRONICOS] Transponder AIS Vector
    [ELETRONICOS] Radar AIS Zen
    [ELETRONICOS] GPS AIS Zen
    [ELETRONICOS] Piloto Automático Simrad Titan Flux Magnum
    [ELETRONICOS] GPS Furuno Swift Leviathan Poseidon
    [ELETRONICOS] Rádio Furuno Zen Swift
    [ELETRONICOS] Radar Garmin Tidal Thrust
    [ELETRONICOS] Sonda Garmin Axis
    [ELETRONICOS] Rádio AIS Oceanic Pulse
    [ELETRONICOS] Rádio Simrad Axis
    [ELETRONICOS] Radar AIS Marlin
    [ELETRONICOS] GPS AIS Nautic Thrust
    [ELETRONICOS] Piloto Automático Furuno Torque Peak
    [ELETRONICOS] GPS AIS Orca Marlin Velocity
    [ELETRONICOS] GPS Garmin Vortex Maré Drift
    [ELETRONICOS] Piloto Automático AIS Core
    [ELETRONICOS] Rádio Simrad Orca
    [ELETRONICOS] Transponder AIS Boost

## 7. Avaliação Qualitativa

Verificamos se as recomendações fazem sentido de negócio:
- Clientes que compraram **eletrônicos** recebem sugestões de outros eletrônicos?
- Clientes de **propulsão** recebem motores compatíveis?
- Evitamos recomendar produtos já comprados?

In [11]:
def avaliar_coerencia(id_client, recomendacoes, df_vendas, df_produtos):
    """Mede a proporção de recomendações na mesma categoria das compras do cliente."""
    if recomendacoes.empty:
        return

    # Categorias já compradas
    prods_comprados = df_vendas[df_vendas['id_client'] == id_client]['id_product'].unique()
    cats_compradas = set(
        df_produtos[df_produtos['code'].isin(prods_comprados)]['category'].unique()
    )

    # Verificar se produto recomendado já foi comprado (não deve aparecer)
    overlap = set(recomendacoes['id_produto']) & set(prods_comprados)
    coerencia = recomendacoes['categoria'].isin(cats_compradas).mean()

    print(f'\n  Cliente {id_client}:')
    print(f'    Categorias do cliente : {cats_compradas}')
    print(f'    Coerência categórica  : {coerencia:.0%} das recomendações')
    print(f'    Itens já comprados repetidos: {len(overlap)} (deve ser 0)')

print('=== Avaliação Qualitativa ===')
for cid, rec in [(cliente_heavy, rec1), (cliente_medio, rec2), (cliente_light, rec3)]:
    avaliar_coerencia(cid, rec, df_vendas, df_produtos)

=== Avaliação Qualitativa ===

  Cliente 15:
    Categorias do cliente : {'ELETRONICOS', 'PROPULSAO', 'ANCORAGEM'}
    Coerência categórica  : 100% das recomendações
    Itens já comprados repetidos: 0 (deve ser 0)

  Cliente 28:
    Categorias do cliente : {'ELETRONICOS', 'PROPULSAO', 'ANCORAGEM'}
    Coerência categórica  : 100% das recomendações
    Itens já comprados repetidos: 0 (deve ser 0)

  Cliente 31:
    Categorias do cliente : {'ELETRONICOS', 'PROPULSAO', 'ANCORAGEM'}
    Coerência categórica  : 100% das recomendações
    Itens já comprados repetidos: 0 (deve ser 0)


In [12]:
# Distribuição de categorias nas recomendações (visão geral)
todas_rec = pd.concat([rec1, rec2, rec3], ignore_index=True)
print('\nDistribuição de categorias nas recomendações geradas:')
print(todas_rec['categoria'].value_counts())


Distribuição de categorias nas recomendações geradas:
categoria
ELETRONICOS    7
ANCORAGEM      6
PROPULSAO      3
Name: count, dtype: int64


## 8. Empacotando e Salvando o Modelo

In [13]:
# Salvar tudo que é necessário para usar o modelo em produção
artefato = {
    'modelo': modelo,
    'matriz': matriz,
    'df_produtos': df_produtos,
    'df_clientes': df_clientes,
    'db_path': DB_PATH,
    'n_neighbors': N_NEIGHBORS,
    'metrica': 'cosine',
    'versao': '1.0'
}

MODEL_FILE = os.path.join(MODELS_PATH, 'modelo_recomendacao.pkl')
joblib.dump(artefato, MODEL_FILE)

size_kb = os.path.getsize(MODEL_FILE) / 1024
print(f'Modelo salvo em : {MODEL_FILE}')
print(f'Tamanho         : {size_kb:.1f} KB')

Modelo salvo em : /home/vinic/Documentos/Projetos/LH_Nautical/models/modelo_recomendacao.pkl
Tamanho         : 137.6 KB


In [14]:
# Verificação: carregar o modelo e fazer uma predição
artefato_carregado = joblib.load(MODEL_FILE)
modelo_carregado  = artefato_carregado['modelo']
matriz_carregada  = artefato_carregado['matriz']
prods_carregados  = artefato_carregado['df_produtos']
clis_carregados   = artefato_carregado['df_clientes']

rec_test = recomendar_produtos(
    cliente_heavy,
    modelo_carregado, matriz_carregada,
    prods_carregados, clis_carregados,
    n_recomendacoes=3
)
print('Verificação OK — modelo carregado com sucesso!')
print(rec_test)

Verificação OK — modelo carregado com sucesso!
   id_produto                                        nome  categoria     score
0         116  Cabo de Nylon Delta Velocity Oceanic Abyss  ANCORAGEM  8.078079
1         144         Boia de Arqueamento Bruce Barracuda  ANCORAGEM  7.279054
2          85          Motor de Popa Parsun Impulse 162HP  PROPULSAO  7.278137
